---
title: Assignment-3
author: Siddharth Rajesh
date: 12-03-2026
---

# Defining a function to parse through PTT files and predict the operons

In [1]:
def predict_operons_from_ptt(file_path):
    operons = []            # Initiate an empty list to append all the operons
    current_operon = []     # Initiate an empty list to append the current operon
    previous_end = None     # Store the end of the previous operon
    previous_strand = None  # Store whether the operon is on the +ve strand or on the -ve strand
    
    with open(file_path) as f:
        next(f)
        next(f)
        next(f)     # Skip the first 3 lines of the PTT files
        
        for line in f:                          # Parse through each row
            parts = line.strip().split('\t')    # Splitting the row into columns
            if len(parts) < 5:
                continue
            
            start, end = map(int, parts[0].split('..'))     # Extracting the start and end position of each gene
            strand, gene_name = parts[1], parts[4]          # Extracting the strand and gene name
            
            same_direction = (strand == previous_strand)    # Conditional to check if the operon is on the same strand
            close_enough = (previous_end is not None) and (start - previous_end < 50)   # Conditional to add genes to current operon if the distance between the end position of the operon and a gene not in operon is less than 50
            
            if same_direction and close_enough:
                current_operon.append((start, end, gene_name))          # Add the current gene to the current operon when conditionals are met
            else:
                if len(current_operon) > 1:
                    operons.append(current_operon)                      # Add the current operon to the operon list
                current_operon = [(start, end, gene_name)]              # Creating the current operon
                previous_strand = strand                                # Saving the current strand for future reference 
            
            previous_end = end                                          # Saving the current end of the operon strand
    
    if len(current_operon) > 1:
        operons.append(current_operon)                                  # Adding the current operon to the total operon list
    
    return operons                                                      # Returning the operons

In [2]:
files = {
    "E. coli K12"   : "data/E_coli_K12_MG1655.ptt",         # File paths and the organisms they belong to
    "B. subtilis"   : "data/B_subtilis_168.ptt",
    "Halobacterium" : "data/Halobacterium_NRC1.ptt",
    "Synechocystis" : "data/Synechocystis_PCC6803_uid159873.ptt",
}

for name, path in files.items():                            # Parsing through the dictionary
    operons = predict_operons_from_ptt(path)                # Invoking the functions to predict the operons
    print(f"\n=== {name} — {len(operons)} operons ===")     # Printing the operons for the given organism
    for op in operons:
        start = op[0][0]
        end   = op[-1][1]
        genes = ", ".join(g for _, _, g in op)
        print(f"  {start}..{end}  [{len(op)} genes]  {genes}")



=== E. coli K12 — 780 operons ===
  337..5020  [3 genes]  thrA, thrB, thrC
  10643..11786  [2 genes]  yaaW, yaaI
  16751..16903  [2 genes]  mokC, hokC
  19811..20508  [2 genes]  insB, insA
  21181..25701  [4 genes]  yaaY, ribF, ileS, lspA
  25826..27227  [2 genes]  fkpB, ispH
  29651..34038  [2 genes]  carA, carB
  34781..36162  [2 genes]  caiE, caiD
  39244..41931  [2 genes]  caiA, caiT
  42403..44129  [2 genes]  fixA, fixB
  44180..45750  [2 genes]  fixC, fixX
  47246..49631  [2 genes]  kefF, kefC
  50380..54702  [5 genes]  apaH, apaG, rsmA, pdxA, surA
  59687..63264  [2 genes]  rluA, rapA
  66835..70048  [2 genes]  araA, araB
  72229..75480  [3 genes]  thiQ, thiP, thiB
  78848..83529  [4 genes]  leuD, leuC, leuB, leuA
  85630..87848  [2 genes]  ilvI, ilvH
  89634..100711  [10 genes]  mraZ, rsmH, ftsL, ftsI, murE, murF, mraY, murD, ftsW, murG
  100765..105244  [4 genes]  murC, ddlB, ftsQ, ftsA
  111649..113219  [3 genes]  yacG, yacF, coaE
  114522..117549  [3 genes]  hofC, hofB, ppd

# Function to predict Operons from Crop Microbiome GFF file

In [3]:
from collections import defaultdict

def predict_operons_gff(filepath):
    # Step 1: collect genes per contig, sorted by start position
    contig_genes = defaultdict(list)            # Creates a dictionary with a list as value for every key

    with open(filepath) as f:
        for line in f:                          # Iterating over the rows
            if line.startswith('#'):            # Skipping the first row
                continue
            parts = line.strip().split('\t')    # Splitting the rows into columns
            if len(parts) < 7:
                continue

            contig = parts[0]                   # Extracting the contigs from column 1
            start  = int(parts[3])              # Extracting the start position of contig
            end    = int(parts[4])              # Extracting the end position of contig
            strand = parts[6]                   # Extracting the strand in which the contig is located (+ve or -ve strand)

            # extract locus_tag from attributes column
            gene_id = parts[8] if len(parts) > 8 else f"{start}..{end}"     # Extracting the gene_ids that are located in the specific contig
            for attr in parts[8].split(';'):
                if attr.startswith('locus_tag='):
                    gene_id = attr.split('=', 1)[1]                         # Extracting the gene name/id located on the contig
                    break

            contig_genes[contig].append((start, end, strand, gene_id))      # Appending the contig as well as the genes to the dictionary

    for contig in contig_genes:                         # Iterating over the contigs in the dictionary
        contig_genes[contig].sort(key=lambda x: x[0])   # Sorting the contigs according to their start position

    # Step 2: slide through each contig and group into operons
    operons = []                # Initialising a list to store the operons

    for contig, genes in contig_genes.items():
        current_operon = []     # List to store the current operon
        prev_end    = None      # Stores the end position of previous operon
        prev_strand = None      # Stores the strand on which the previous operon is located

        for start, end, strand, gene_id in genes:       
            same_direction = (strand == prev_strand)                                # Conditional to check if the operon is on the same strand
            close_enough   = (prev_end is not None) and (start - prev_end < 50)     # Conditional to add genes to current operon if the distance between the end position of current operon and a gene not in operon is less than 50

            if same_direction and close_enough:
                current_operon.append((start, end, gene_id))            # Add the current gene to an existing operon if the conditions are met
            else:
                if len(current_operon) > 1:
                    operons.append((contig, current_operon))            # Adding the contigs as well as the operon situated on the contig
                current_operon = [(start, end, gene_id)]                # Creating a new operon
                prev_strand = strand                                    # Storing the strand of the new operon

            prev_end = end                                              # Storing the end position of the operon

        if len(current_operon) > 1:
            operons.append((contig, current_operon))                    # Adding the contigs as well as the operon situated on the contig

    return operons                                                      # Returning the operons


In [4]:
operons = predict_operons_gff("data/2088090036.gff")
print(f"\n=== Hoatzin Crop Microbiome — {len(operons)} operons ===")
for contig, op in operons:
    start = op[0][0]
    end   = op[-1][1]
    genes = ", ".join(g for _, _, g in op)
    print(f"  [{contig}]  {start}..{end}  [{len(op)} genes]  {genes}")



=== Hoatzin Crop Microbiome — 758 operons ===
  [HCP21_2940]  1..371  [2 genes]  HCP21_00000140, HCP21_00000150
  [HCP21_118_]  42..3247  [3 genes]  HCP21_00000640, HCP21_00000650, HCP21_00000660
  [HCP21_6018]  201..584  [2 genes]  HCP21_00001280, HCP21_00001290
  [HCP21_2399]  1123..2241  [2 genes]  HCP21_00001740, HCP21_00001750
  [HCP21_2957]  62..554  [2 genes]  HCP21_00001890, HCP21_00001900
  [HCP21_8019]  3..1220  [2 genes]  HCP21_00001950, HCP21_00001960
  [HCP21_2422]  2..2282  [2 genes]  HCP21_00002160, HCP21_00002170
  [HCP21_9479]  1..487  [2 genes]  HCP21_00002230, HCP21_00002240
  [HCP21_5152]  2..420  [2 genes]  HCP21_00002520, HCP21_00002530
  [HCP21_3570]  3..1092  [3 genes]  HCP21_00002590, HCP21_00002600, HCP21_00002610
  [HCP21_1959]  2..444  [2 genes]  HCP21_00003150, HCP21_00003160
  [HCP21_4442]  3..1310  [2 genes]  HCP21_00003220, HCP21_00003230
  [HCP21_5777]  647..1316  [3 genes]  HCP21_00003280, HCP21_00003290, HCP21_00003300
  [HCP21_4436]  1..486  [2 gene